# Open Images V7 — Dataset Downloader

Downloads 40 images per category from Open Images V7 for a computer vision project using SAM (Segment Anything Model).

## Categories
- Dog
- Train
- Mobile phone

## Output
```
training_data/
├── Dog/           (40 images)
├── Train/         (40 images)
└── Mobile_phone/  (40 images)
```

## Tools
- **FiftyOne** — selective downloading from Open Images without pulling the full 561GB dataset
- **Open Images V7** — source dataset by Google, free under CC BY 4.0
- **Google Drive** — persistent storage across Colab sessions

## Notes
- Full resolution images (up to 1024px), suitable for SAM
- Seed 42 used for reproducibility
- Images are committed to the repo — re-running is not necessary
- Citation: Kuznetsova et al., "The Open Images Dataset V7", 2023

**Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Install**

In [ ]:
!pip install fiftyone

**Downloads**

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
import os, shutil, random

CATEGORIES = ["Dog", "Train", "Mobile phone"]
COUNT_PER_CAT = 40
DOWNLOAD_DIR = '/content/drive/MyDrive/training_data'
random.seed(42)

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

for cat in CATEGORIES:
    cat_dir = os.path.join(DOWNLOAD_DIR, cat.replace(" ", "_"))
    os.makedirs(cat_dir, exist_ok=True)

    # Skip if already complete
    existing = len(os.listdir(cat_dir))
    if existing >= COUNT_PER_CAT:
        print(f"{cat}: already complete ({existing} images), skipping")
        continue

    print(f"Downloading {cat}...")
    dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        label_types=["detections"],
        classes=[cat],
        max_samples=COUNT_PER_CAT,
        seed=42,
        shuffle=True,
        dataset_name=f"open-images-{cat.replace(' ', '-')}",
    )

    count = 0
    for sample in dataset:
        if count >= COUNT_PER_CAT:
            break
        dest = os.path.join(cat_dir, f"{cat.replace(' ', '_')}_{count:04d}.jpg")
        shutil.copy(sample.filepath, dest)
        count += 1

    print(f"  Saved {count} images → {cat_dir}")

print("Done!")

**Verify**

In [ ]:
import os
for cat in ["Dog", "Train", "Mobile_phone"]:
    path = f'/content/drive/MyDrive/training_data/{cat}'
    if os.path.exists(path):
        print(f"{cat}: {len(os.listdir(path))} images")
    else:
        print(f"{cat}: nothing saved yet")